# Multi-Dataset Reflectometry Fitting - Teaching Notebook

This notebook is structured as guided teaching material for **simultaneous fitting** of multiple reflectivity datasets using [EasyReflectometry](https://github.com/easyscience/EasyReflectometryLib).

The input file `reflectivity_geomgrid.ort` contains **4 datasets** (data_set: 0-3) from an ESS Estia instrument simulation. All datasets share one structural model but each covers its own Q-range.

Learning goals:
- Build and fit a baseline layer model from physical reasoning.
- Critically evaluate fit quality and propose improvements.
- Compare a simple model against a more realistic interlayer model.

## 1. Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from easyscience.fitting import AvailableMinimizers

from easyreflectometry.calculators import CalculatorFactory
from easyreflectometry.data import load
from easyreflectometry.fitting import MultiFitter
from easyreflectometry.model import Model, PercentageFwhm
from easyreflectometry.sample import Layer, Material, Multilayer, Sample

%matplotlib inline

## 2. Load Experimental Data

Load the `.ort` file and inspect the available datasets. Each dataset has its own Q-range and reflectivity values.

In [ ]:
file_path = 'reflectivity_geomgrid.ort'
data = load(file_path)

dataset_keys = sorted(data['data'].keys())
num_datasets = len(dataset_keys)

print(f'Loaded {num_datasets} datasets from {file_path}:\n')
for k in dataset_keys:
    coord_k = k.replace('R_', 'Qz_')
    qz = data['coords'][coord_k].values
    r = data['data'][k].values
    print(f'  {k}: {len(r)} points, Qz range [{qz.min():.4f}, {qz.max():.4f}] 1/\u00c5')

## 3. Materials and Layer-Order Hypothesis

Before looking at any model-building code, answer this:

- Given materials **Air**, **Ni**, **Si**, what is the physical stack order?
- Which one is the **superphase**?
- Which one is the **thin film**?
- Which one is the **substrate**?

Expected baseline interpretation:
- Air = superphase
- Ni = thin film
- Si = substrate

We define these three materials for the baseline model:

| Material | SLD (10⁻⁶ Å⁻²) | Description |
|----------|-----------------|-------------|
| Air      | 0.0             | Superphase  |
| Ni       | 8.746           | Thin film   |
| Si       | 2.07            | Substrate   |

In [ ]:
air = Material(sld=0.0, isld=0.0, name='Air')
ni = Material(sld=8.746, isld=0.0, name='Ni')
si = Material(sld=2.07, isld=0.0, name='Si')

## 4. Define Layers (Try First, Then Expand)

Task:
- Create layer objects for Air, Ni, and Si.
- Choose initial guesses for thickness and roughness.
- Keep Air and Si semi-infinite (`thickness = 0`).


In [ ]:
air_layer = Layer(material=air, thickness=0, roughness=0, name='Air Superphase')
ni_layer = Layer(material=ni, thickness=183.9, roughness=1.3, name='Ni_layer')
si_layer = Layer(material=si, thickness=0, roughness=6.2, name='Si Subphase')

## 5. Build the Model (Try First, Then Expand)

Task:
- Assemble a `Sample` using the stack Air | Ni | Si.
- Wrap it in a `Model` with scale, background, and resolution.
- Connect the calculator interface.

Hints for physical meaning:
- **Scale factor** — overall intensity scaling  
- **Background** — incoherent/instrumental background  
- **Resolution** — Gaussian smearing (3% FWHM)  

A single model instance is shared across all datasets.


In [ ]:
ni_assembly = Multilayer(ni_layer, name='Ni Assembly')
sample = Sample(Multilayer(air_layer), ni_assembly, Multilayer(si_layer), name='NiSi')

model = Model(
    sample=sample,
    scale=0.4,
    background=5.4e-7,
    resolution_function=PercentageFwhm(3),
    name='NiSi_Model',
)
model.interface = CalculatorFactory()

## 6. Set Free Parameters (Try First, Then Expand)

Task:
- Decide which parameters should vary in the baseline model.
- Propose physically sensible bounds.

Start with: Ni thickness, Ni roughness, Si roughness, scale, and background.


In [ ]:
# Ni layer thickness
ni_layer.thickness.fixed = False
ni_layer.thickness.min = 50.0
ni_layer.thickness.max = 500.0

# Ni layer roughness
ni_layer.roughness.fixed = False
ni_layer.roughness.min = 0.0
ni_layer.roughness.max = 20.0

# Si substrate roughness
si_layer.roughness.fixed = False
si_layer.roughness.min = 0.0
si_layer.roughness.max = 30.0

# Model scale
model.scale.fixed = False
model.scale.min = 0.1
model.scale.max = 2.0

# Model background
model.background.fixed = False
model.background.min = 1e-10
model.background.max = 1e-4

## 7. Run the Multi-Dataset Fit

The same model object is passed once per dataset to `MultiFitter`, matching the ERA GUI approach where each experiment keeps its own Q-range but shares the model's structural parameters.

We use the **Levenberg–Marquardt** minimizer from LMFit and weight each point by $1/\sigma$.

In [ ]:
fitter = MultiFitter(*([model] * num_datasets))
fitter.easy_science_multi_fitter.switch_minimizer(AvailableMinimizers.LMFit_leastsq)

# Prepare per-dataset arrays
refl_nums = sorted(k[3:] for k in data['coords'].keys() if k.startswith('Qz'))
x_data = []
y_data = []
weights = []

for rid in refl_nums:
    x_vals = data['coords'][f'Qz_{rid}'].values
    y_vals = data['data'][f'R_{rid}'].values
    variances = data['data'][f'R_{rid}'].variances
    # Mask zero-variance points
    valid = variances > 0
    x_data.append(x_vals[valid])
    y_data.append(y_vals[valid])
    weights.append(1.0 / np.sqrt(variances[valid]))

print(f'Fitting single model to {num_datasets} datasets simultaneously...')
fit_results = fitter.easy_science_multi_fitter.fit(x_data, y_data, weights=weights)

## 8. Fit Results

Print the goodness-of-fit metric and the refined parameter values.

Discussion prompts:
- Does the fit capture all features across the full Q-range?
- Which parts of the curve show the largest mismatch?
- What missing physics might explain residual structure?

In [ ]:
# Compute reduced chi-squared across all datasets
total_chi2 = sum(r.chi2 for r in fit_results)
total_points = sum(np.size(r.x) for r in fit_results)
n_params = fit_results[0].n_pars
reduced_chi2 = total_chi2 / (total_points - n_params)

print(f'Reduced \u03c7\u00b2 = {reduced_chi2:.4f}')
print(f'Fit converged: {fit_results[0].success}\n')

print('=== Fitted parameters ===')
print(f'  Ni SLD        = {ni_layer.material.sld.value:.4f} \u00d7 10\u207b\u2076 \u00c5\u207b\u00b2 (fixed)')
print(f'  Ni thickness  = {ni_layer.thickness.value:.2f} \u00c5')
print(f'  Ni roughness  = {ni_layer.roughness.value:.2f} \u00c5')
print(f'  Si roughness  = {si_layer.roughness.value:.2f} \u00c5')
print(f'  Scale         = {model.scale.value:.4f}')
print(f'  Background    = {model.background.value:.2e}')

## 9. Visualisation

### Reflectivity curves
Experimental data (points with error bars) vs. fitted model (solid lines) for all datasets on a log scale.

### SLD profile
The scattering length density as a function of depth — identical for all datasets since they share one model.

In [ ]:
color_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']

# SLD profile (same for all datasets)
sld_profile = model.interface().sld_profile(model.unique_name)
z_sld, sld_vals = sld_profile[0], sld_profile[1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Reflectivity ---
for i, rid in enumerate(refl_nums):
    qz = data['coords'][f'Qz_{rid}'].values
    r = data['data'][f'R_{rid}'].values
    sr = np.sqrt(data['data'][f'R_{rid}'].variances)
    color = color_cycle[i % len(color_cycle)]

    ax1.errorbar(
        qz, r, yerr=sr,
        fmt='.', markersize=3, capsize=0, color=color,
        alpha=0.5, label=f'Data {rid}',
    )
    r_model = model.interface.fit_func(qz, model.unique_name)
    ax1.plot(qz, r_model, '-', color=color, linewidth=1.5, label=f'Fit {rid}')

ax1.set_yscale('log')
ax1.set_xlabel(r'$Q_z$ ($\AA^{-1}$)')
ax1.set_ylabel('Reflectivity')
ax1.set_title(f'Multi-dataset fit (red. $\\chi^2$ = {reduced_chi2:.2f})')
ax1.legend(fontsize=8, loc='upper right')
ax1.grid(True, which='both', linestyle=':', alpha=0.4)

# --- SLD profile ---
ax2.plot(z_sld, sld_vals, '-', color='black', linewidth=1.5)
ax2.set_xlabel(r'z ($\AA$)')
ax2.set_ylabel(r'SLD ($10^{-6}\,\AA^{-2}$)')
ax2.set_title('SLD profile')
ax2.grid(True, linestyle=':', alpha=0.4)

fig.tight_layout()
plt.show()

---

# Part 2: Guided Improvement Challenge

The simple **Si | Ni | Air** model assumes ideal, sharp interfaces.

Before revealing a refined model, propose improvements:
- What could form between **Ni and Air** after exposure/processing?
- What could form between **Si and Ni** due to intermixing or reaction?

Possible hypothesis:
- A nickel oxide-like layer (**NiOx**) near the top interface
- A nickel silicide-like intermixed layer (**NiSi**) near the substrate interface

We now extend the model and check whether reduced chi-squared improves.

## 10. Define Interlayer Materials

One concrete implementation uses:
- **NiOx** represented here with a NiO-like SLD
- **NiSi** as an intermixed silicide layer

| Material | SLD (10⁻⁶ Å⁻²) | Description |
|----------|-----------------|-------------|
| NiO      | 6.7             | Nickel oxide at the Ni–air interface |
| NiSi     | 5.9             | Nickel silicide at the Si–Ni interface |

In [ ]:
nio = Material(sld=6.7, isld=0.0, name='NiO')
nisi = Material(sld=5.9, isld=0.0, name='NiSi')

## 11. Define Interlayer Layers and Rebuild the Sample (Try First, Then Expand)

Task:
- Add interlayers at the two interfaces motivated above.
- Rebuild the sample with a physically consistent order.

The full stack from substrate to superphase is:

**Si -> NiSi -> Ni -> NiO -> Air**


In [ ]:
# Interlayer layers
nio_layer = Layer(material=nio, thickness=15, roughness=2, name='NiO_layer')
nisi_layer = Layer(material=nisi, thickness=35, roughness=2, name='NiSi_layer')

# Fresh copies of core layers (so Part 1 results are preserved)
ni_layer2 = Layer(material=Material(sld=8.746, isld=0.0, name='Ni2'), thickness=135., roughness=2., name='Ni_layer2')
si_layer2 = Layer(material=Material(sld=2.07, isld=0.0, name='Si2'), thickness=0, roughness=4.0, name='Si2 Subphase')
air_layer2 = Layer(material=Material(sld=0.0, isld=0.0, name='Air2'), thickness=0, roughness=0, name='Air2 Superphase')

# Sample: air | NiO | Ni | NiSi | Si  (top to bottom)
sample2 = Sample(
    Multilayer(air_layer2),
    Multilayer(nio_layer, name='NiO Assembly'),
    Multilayer(ni_layer2, name='Ni Assembly 2'),
    Multilayer(nisi_layer, name='NiSi Assembly'),
    Multilayer(si_layer2),
    name='NiSi_interlayers',
)

model2 = Model(
    sample=sample2,
    scale=0.4,
    background=5.4e-7,
    resolution_function=PercentageFwhm(3),
    name='NiSi_Interlayer_Model',
)
model2.interface = CalculatorFactory()

## 12. Set Free Parameters for Interlayer Model (Try First, Then Expand)

Task:
- Decide which new interlayer parameters should be free.
- Set realistic bounds to avoid unphysical solutions.

Compared with Part 1, we now also refine:

- **NiO**: SLD, thickness, roughness  
- **NiSi**: SLD, thickness, roughness

This adds 6 extra degrees of freedom (11 total vs. 5 before).


In [ ]:
# NiO: sld, thickness, roughness
nio_layer.material.sld.fixed = False
nio_layer.material.sld.value = 6.7
nio_layer.material.sld.min = 0.0
nio_layer.material.sld.max = 15.0

nio_layer.thickness.fixed = False
nio_layer.thickness.value = 15.0
nio_layer.thickness.min = 0.0
nio_layer.thickness.max = 100.0

nio_layer.roughness.fixed = False
nio_layer.roughness.value = 2.0
nio_layer.roughness.min = 0.0
nio_layer.roughness.max = 20.0

# NiSi: sld, thickness, roughness
nisi_layer.material.sld.fixed = False
nisi_layer.material.sld.value = 5.9
nisi_layer.material.sld.min = 0.0
nisi_layer.material.sld.max = 15.0

nisi_layer.thickness.fixed = False
nisi_layer.thickness.value = 35.0
nisi_layer.thickness.min = 0.0
nisi_layer.thickness.max = 100.0

nisi_layer.roughness.fixed = False
nisi_layer.roughness.value = 2.0
nisi_layer.roughness.min = 0.0
nisi_layer.roughness.max = 20.0

# Ni layer
ni_layer2.thickness.fixed = False
ni_layer2.thickness.value = 135.
ni_layer2.thickness.min = 50.0
ni_layer2.thickness.max = 500.0

ni_layer2.roughness.fixed = False
ni_layer2.roughness.value = 2.0
ni_layer2.roughness.min = 0.0
ni_layer2.roughness.max = 20.0

# Si roughness
si_layer2.roughness.fixed = False
si_layer2.roughness.value = 4.0
si_layer2.roughness.min = 0.0
si_layer2.roughness.max = 30.0

model2.background.max = 1e-4

# Scale and background
model2.background.min = 1e-10

model2.scale.fixed = False
model2.background.fixed = False

model2.scale.min = 0.1
model2.scale.max = 2.0

## 13. Fit the Interlayer Model

Same procedure as Part 1 — we pass the model once per dataset to `MultiFitter` and use Levenberg–Marquardt.

In [ ]:
fitter2 = MultiFitter(*([model2] * num_datasets))
fitter2.easy_science_multi_fitter.switch_minimizer(AvailableMinimizers.LMFit_leastsq)

print(f'Fitting Si|NiSi|Ni|NiO|air model to {num_datasets} datasets simultaneously...')
fit_results2 = fitter2.easy_science_multi_fitter.fit(x_data, y_data, weights=weights)

## 14. Interlayer Fit Results & χ² Comparison

Print the refined parameters and compare reduced chi-squared for the two models.

Discussion prompts:
- Did the interlayer model improve the fit significantly?
- Is the improvement large enough to justify the added complexity?
- Which parameters are physically interpretable, and which may be compensating each other?

In [ ]:
# Reduced chi-squared for the interlayer model
total_chi2_2 = sum(r.chi2 for r in fit_results2)
total_points_2 = sum(np.size(r.x) for r in fit_results2)
n_params_2 = fit_results2[0].n_pars
reduced_chi2_2 = total_chi2_2 / (total_points_2 - n_params_2)

print(f'Reduced χ² = {reduced_chi2_2:.4f}')
print(f'Fit converged: {fit_results2[0].success}\n')

print('=== Fitted parameters (interlayer model) ===')
print(f'  NiO SLD        = {nio_layer.material.sld.value:.4f} × 10⁻⁶ Å⁻²')
print(f'  NiO thickness  = {nio_layer.thickness.value:.2f} Å')
print(f'  NiO roughness  = {nio_layer.roughness.value:.2f} Å')
print(f'  NiSi SLD       = {nisi_layer.material.sld.value:.4f} × 10⁻⁶ Å⁻²')
print(f'  NiSi thickness = {nisi_layer.thickness.value:.2f} Å')
print(f'  NiSi roughness = {nisi_layer.roughness.value:.2f} Å')
print(f'  Ni thickness   = {ni_layer2.thickness.value:.2f} Å')
print(f'  Ni roughness   = {ni_layer2.roughness.value:.2f} Å')
print(f'  Si roughness   = {si_layer2.roughness.value:.2f} Å')
print(f'  Scale          = {model2.scale.value:.4f}')
print(f'  Background     = {model2.background.value:.2e}')

print('\n=== χ² comparison ===')
print(f'  Si|Ni|air           : red. χ² = {reduced_chi2:.4f}')
print(f'  Si|NiSi|Ni|NiO|air : red. χ² = {reduced_chi2_2:.4f}')
improvement = (reduced_chi2 - reduced_chi2_2) / reduced_chi2 * 100
print(f'  Improvement         : {improvement:.1f}%')

## 15. Final Comparison and Reflection

Use the plots and metrics to evaluate model quality.

Final teaching questions:
- What specific improvements do you observe in reflectivity and SLD behaviour?
- What additional complexity could be added next (e.g., graded interfaces, multiple oxides, hydration, lateral inhomogeneity)?
- What data or constraints would be needed to justify more complex structures?

In [ ]:
sld_profile2 = model2.interface().sld_profile(model2.unique_name)
z_sld2, sld_vals2 = sld_profile2[0], sld_profile2[1]

fig2, (ax3, ax4) = plt.subplots(1, 2, figsize=(14, 6))

# --- Reflectivity ---
for i, rid in enumerate(refl_nums):
    qz = data['coords'][f'Qz_{rid}'].values
    r = data['data'][f'R_{rid}'].values
    sr = np.sqrt(data['data'][f'R_{rid}'].variances)
    color = color_cycle[i % len(color_cycle)]

    ax3.errorbar(
        qz, r, yerr=sr,
        fmt='.', markersize=3, capsize=0, color=color,
        alpha=0.5, label=f'Data {rid}',
    )
    r_model2 = model2.interface.fit_func(qz, model2.unique_name)
    ax3.plot(qz, r_model2, '-', color=color, linewidth=1.5, label=f'Fit {rid}')

ax3.set_yscale('log')
ax3.set_xlabel(r'$Q_z$ ($\AA^{-1}$)')
ax3.set_ylabel('Reflectivity')
ax3.set_title(f'Si|NiSi|Ni|NiO|air fit (red. $\\chi^2$ = {reduced_chi2_2:.2f})')
ax3.legend(fontsize=8, loc='upper right')
ax3.grid(True, which='both', linestyle=':', alpha=0.4)

# --- SLD profile comparison ---
ax4.plot(z_sld2, sld_vals2, '-', color='black', linewidth=1.5, label='Interlayer model')
ax4.plot(z_sld, sld_vals, '--', color='gray', linewidth=1.0, label='Simple model')
ax4.set_xlabel(r'z ($\AA$)')
ax4.set_ylabel(r'SLD ($10^{-6}\,\AA^{-2}$)')
ax4.set_title('SLD profile comparison')
ax4.legend()
ax4.grid(True, linestyle=':', alpha=0.4)

fig2.tight_layout()
plt.show()